# Race condition scenario 1

These two critical regions access the count2 member variable of the device. At the time of entering the CR sem1 is held, and in the case of the release function, sem2 is also held. 

In normal execution, there should be no race condition here because both critical regions are controlled by semaphore 1, requiring that only one process hold the semaphore at a time, regardless of semaphore 2. However, there is no error handling on the down interruptible call. Thus, it is possible that a process executing critical region 1 gets blocked, receives a signal, then continues to enter the critical region without acquiring semaphore 1. This would cause the count to prematurely increment which could cause unexpected behavior in other branch conditions. 

In [ ]:
    // ---------------- Critical Region 1: open mode 2 -------------------------------//
    down_interruptible(&devc->sem1);
    else if (devc->mode == MODE2) {
        devc->count2++;
    }
    // -----------------------------------------------------------------// 

    // ------------ Critical Region 2: release mode 2 ----------------------------//
    down_interruptible(&devc->sem1);
    else if (devc->mode == MODE2) {
        devc->count2--;
        if (devc->count2 == 1)
            wake_up_interruptible(&(devc->queue2));
    }
    // -----------------------------------------------------------------// 

# Race condition scenario 2:
In mode 1, Sem 1 is released prior to accessing the ramdisk in both the read/write functions. 

The driver assumes that there will only be one process accessing the critical region at a time because sem2 is held. However, it is possible for a process to bypass the semaphore since down_interruptible is not handled.

This may result in two processes accessing the critical region and data being invalidated. For example, a process could read from the ramdisk, check some condition on the data read, then a second process could write over that data.

In [ ]:
    // ---------------- Critical Region 1: read -------------------------------//
    down_interruptible(&devc->sem1);
    else {
       if (*f_pos + count > ramdisk_size) {
          printk("Trying to read past end of buffer!\n");
          up(&devc->sem1);
          return ret;
       }
       ret = count - copy_to_user(buf, devc->ramdisk, count);
       *f_pos += ret;
       up(&devc->sem1);
    }
    // -----------------------------------------------------------------// 

    // ------------ Critical Region 2: write ----------------------------//
    down_interruptible(&devc->sem1);
    else {
       if (*f_pos + count > ramdisk_size) {
          printk("Trying to read past end of buffer!\n");
          up(&devc->sem1);
          return ret;
       }
       ret = count - copy_from_user(devc->ramdisk, buf, count);
       *f_pos += ret;
       up(&devc->sem1);
    }
    // -----------------------------------------------------------------// 

# Race Condition Scenario 3

> Critical Region 1: `e2_release()` in Mode 2
- `sem1` is intended to be held after `down_interruptible(&devc->sem1)`.
- Shared state accessed: `devc->count2` (decrement), and `devc->queue2` via `wake_up_interruptible` when `count2 == 1`.
- `sem2` is **not** used in this region.

> Critical Region 2: `e2_ioctl()` case `E2_IOCMODE1`
- `sem1` is intended to be held during checks/updates of `count2`, `count1`, and `mode`.
- Shared state accessed: `devc->count2` in the wait loop and decrement, `devc->mode` (set to `MODE1`), and `devc->count1` (increment).
- The code temporarily releases `sem1` while waiting (`wait_event_interruptible`), then re-acquires it before touching shared counters again.
- `sem2` is acquired near the end (`down_interruptible(&devc->sem2)`) while still holding `sem1`.

> Is this a race condition?
- **Under normal successful lock acquisition, this is not a true data race on `count2`** because both regions serialize `count2` with `sem1`.
- **It can become a race** because return values of `down_interruptible(...)` are ignored. If interrupted by a signal, execution may continue without owning `sem1`, causing unsynchronized reads/writes to `count2`, `count1`, and `mode`.
- So the root issue is: **missing error handling on semaphore acquisition can invalidate mutual exclusion**.

> Suggested code-review conclusion
- Describe this as a **conditional race vulnerability**: synchronization is correct in design, but broken in failure paths.
- Also note that lock ordering (`sem1` then `sem2` in ioctl path) should be reviewed separately for deadlock risk.

In [ ]:
    // ---------------- Critical Region 1: release Mode 2 -------------------------------//
    down_interruptible(&devc->sem1);
    else if (devc->mode == MODE2) {
        devc->count2--;
        if (devc->count2 == 1)
            wake_up_interruptible(&(devc->queue2));
    }
    up(&devc->sem1);
    // -------------------------------------------------------------------------------//

    // ---------------- Critical Region 2: IOCTL Mode 1 -------------------------------//
    down_interruptible(&(devc->sem1));
    if (devc->count2 > 1) {
        while (devc->count2 > 1) {
        up(&devc->sem1);
        wait_event_interruptible(devc->queue2, (devc->count2 == 1));
        down_interruptible(&devc->sem1);
        }
    }
    devc->mode = MODE1;
    devc->count2--;
    devc->count1++;
    down_interruptible(&devc->sem2);
    up(&devc->sem1);
    // -------------------------------------------------------------------------------//


# Race Condition Scenario 4: ioctl wait loop and release count decrement

No race condition here.

Locks held: 
    - ioctl holds the lock while checking count1, then releases before sleeping.
    - release holds sem1 while decrementing the count and wakes up sleeping process.

Data accessed:

    - count1 (protected by sem1)
    - queue1 (thread safe)

Analysis: There is no race condition because locks are properly used to protect shared data, and there is no concern for missed wakeups because wait_event_interruptible does an additional check for the count1.

In [ ]:
// ---------------- Critical Region 1: ioctl mode 2 -------------------------------//
down_interruptible(&(devc->sem1));
while (devc->count1 > 1) {
    up(&devc->sem1);
    wait_event_interruptible(devc->queue1, (devc->count1 == 1));
    down_interruptible(&devc->sem1);
}
// -------------------------------------------------------------------------------//

// ---------------- Critical Region 2: release mode 1 -------------------------------//
    down_interruptible(&devc->sem1);
    if (devc->mode == MODE1) {
        devc->count1--;
        if (devc->count1 == 1)
            wake_up_interruptible(&(devc->queue1));
	up(&devc->sem2);
    }
    up(&devc->sem1);
// -------------------------------------------------------------------------------//



# Deadlock Scenario 1
Wait_event_interruptible called with second thread blocks and no way for the event to update.

Process 1 opens in Mode1

Process 2 opens in Mode1 and blocks while trying to aquire sem2

Process 1 calls IOCTL to switch to mode 2 (count1 > 1) and sleeps on wait_event_interruptible

Both processes are blocked.

In [ ]:
int e2_open(struct inode *inode, struct file *filp)                             
    {                                                                               
        struct e2_dev *devc = container_of(inode->i_cdev, struct e2_dev, cdev);     
        filp->private_data = devc;                                                  
        down_interruptible(&devc->sem1);                                            
        if (devc->mode == MODE1) {                                                  
            devc->count1++;                                                         
            up(&devc->sem1);                                                        
            down_interruptible(&devc->sem2);                                        
            return 0;                                                               
        }                                                                           
        else if (devc->mode == MODE2) {                                             
            devc->count2++;                                                         
        }                                                                           
        up(&devc->sem1);                                                            
        return 0;                                                                   
    }

// static long e2_ioctl(struct file *filp, unsigned int cmd, unsigned long arg)
switch(cmd) {                                                               
        case E2_IOCMODE2:                                                        
           down_interruptible(&(devc->sem1));                                    
           if (devc->mode == MODE2) {                                            
              up(&devc->sem1);                                                   
              break;                                                             
           }                                                                     
           if (devc->count1 > 1) {                                               
              while (devc->count1 > 1) {                                         
                 up(&devc->sem1);                                                
                 wait_event_interruptible(devc->queue1, (devc->count1 == 1));    
                 down_interruptible(&devc->sem1);                                
              }                                                                  
           }           
        }

# Deadlock scenario 2

Device initialization.

    count 1 = 0

    count 2 = 0

    mode = 2

Process 1 opens the device.

    count 1 = 1

Process 1 switches to mode 2.

    count 1 = 0
    
    count 2 = 1

Process 2 opens the device (mode 2).

    count 2 = 2

Process 1 calls IOCTL to switch to mode 1.

Process 1 blocked in wait loop (count2 > 1)

Porcess 2 calls IOCTL to switch to mode 1.

Process 2 stuck in wait loop (count2 > 1).

Deadlock.


Relevant code: assignment5.c line 174 - 178

# Deadlock Scenario 3: Killing process causes negative counts

Device Initialized.

Process 1 opens device, and process is killed before incrementing count.

Process 1 calls release function, decrements count1 to -1.

    count1++ = -1

Process 2 opens device (mode 1)

    count1++ = 0

    sem2 held (process 2)

Process 3 opens device (mode 1), blocks trying to acquire sem2.

    count1++ = 1
    
    sem2 held (process 2)

Process 2 calls IOCTL to switch to mode 2

    sem2 released

    count2++ = 1

    count1-- = 0

    mode = mode2

    sem2 help (process 3)

Process 2 calls IOCTL to switch to mode 1

    sem1 held (process 2)

    mode = mode1

    count2-- = 0

    count1++ = 1

Process 2 is now blocked trying to acquire sem2

Process 3 calls read, blocks while trying to acquire sem1

Deadlock


Note: If not for the killed process, this still would have entered the deadlock described in scenario 1 due to sem2 not being relese before the waiting loop for ioctl switch to mode 2.

------------------------------------------------------------------------------------------------

# Deadlock scenario 4